In [1]:
!uv pip install httpx

import time
import random

MODE = "simulate"  # "simulate" | "transformers" | "vllm_http"


Using Python 3.13.15 environment at: /usr
Checked 1 package in 181ms


In [2]:
if MODE == "transformers":
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer

    MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"  # non-quantized, ~32k context
    device = "cuda" if torch.cuda.is_available() else "cpu"

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, torch_dtype=torch.float16
    ).to(device)

    max_ctx = getattr(model.config, "max_position_embeddings", None)
    print(f"Using '{MODEL_NAME}' on {device}, max context = {max_ctx}")
    assert max_ctx is None or max_ctx >= 2048, (
        f"{MODEL_NAME} only supports {max_ctx} tokens of context — "
        "the 2048-token sweep step below will fail. Pick a longer-context model."
    )

    def tok(prompt, return_tensors="pt"):
        return tokenizer(prompt, return_tensors=return_tensors).to(device)

    def prompt_of_len(n):
        # Build a prompt that tokenizes to roughly n tokens.
        text = "hello " * n
        ids = tokenizer(text, return_tensors="pt").input_ids[0][:n]
        return tokenizer.decode(ids)

elif MODE == "vllm_http":
    # --- Real inference via an already-running vLLM server --------------
    # Reuses a server you've already validated works (e.g. from the main
    # lab), avoiding the AWQ/quantization pitfalls of loading it a second
    # time through transformers directly.
    import httpx

    BASE_URL = "http://localhost:8000"
    MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct-AWQ"  # must match --model on the server
    _client = httpx.Client(timeout=120.0)

    class _RealModel:
        def generate(self, input_ids, max_new_tokens=32):
            body = {
                "model": MODEL_NAME,
                "messages": [{"role": "user", "content": input_ids["prompt"]}],
                "max_tokens": max_new_tokens,
                "temperature": 0.0,
                "stream": False,
            }
            r = _client.post(f"{BASE_URL}/v1/chat/completions", json=body)
            r.raise_for_status()
            return r.json()

    model = _RealModel()

    def tok(prompt, return_tensors="pt"):
        return {"input_ids": {"prompt": prompt}}

    def prompt_of_len(n):
        return ("hello " * n).strip()

    print(f"Using vLLM server at {BASE_URL} for model '{MODEL_NAME}'")

else:
    # --- Simulated path (no GPU needed) ----------------------------------
    # Calibrated so the broken (no warm-up) sweep below reproduces the
    # confirmed real run cited in this lab: 128: 1.0744s, 512: 0.0989s,
    # 2048: 0.2632s.
    class _FakeTensor:
        def __init__(self, length):
            self.shape = (1, length)

    class FakeModel:
        """Stand-in for model.generate() that reproduces the same
        cold-start bug real GPU inference has: the FIRST generate() call
        in a session pays a one-time CUDA-context / kernel-compilation
        cost that has nothing to do with prompt length."""

        _warm = False

        COLD_START_COST = 1.016575
        BASE_COST = 0.012133
        PER_PROMPT_TOKEN_COST = 0.00010697
        PER_GEN_TOKEN_COST = 0.001
        JITTER = 0.003

        def generate(self, input_ids, max_new_tokens=32):
            context_len = input_ids.shape[-1]
            cost = (
                self.BASE_COST
                + self.PER_PROMPT_TOKEN_COST * context_len
                + self.PER_GEN_TOKEN_COST * max_new_tokens
                + random.uniform(0, self.JITTER)
            )
            if not FakeModel._warm:
                cost += self.COLD_START_COST
                FakeModel._warm = True
            time.sleep(cost)
            return [0] * max_new_tokens

        @classmethod
        def reset_session(cls):
            """Simulates restarting the Python runtime / a fresh vLLM
            server: all warm-up state is lost."""
            cls._warm = False

    def tok(prompt, return_tensors="pt"):
        return {"input_ids": _FakeTensor(len(prompt.split()))}

    def prompt_of_len(n):
        return " ".join(["token"] * n)

    model = FakeModel()
    random.seed(7)
    print("Using simulated FakeModel (no GPU required)")


Using simulated FakeModel (no GPU required)


In [3]:
if MODE == "simulate":
    FakeModel.reset_session()  # fresh session, matching a real cold process/server

results = {}
for context in [128, 512, 2048]:
    prompt = prompt_of_len(context)
    t0 = time.time()
    out = model.generate(**tok(prompt, return_tensors="pt"), max_new_tokens=32)
    results[context] = time.time() - t0
    print(context, results[context])


128 1.0755231380462646
512 0.0995485782623291
2048 0.2654101848602295


In [4]:
if MODE == "simulate":
    FakeModel.reset_session()  # fresh session again

results_reordered = {}
for context in [2048, 512, 128]:
    prompt = prompt_of_len(context)
    t0 = time.time()
    out = model.generate(**tok(prompt, return_tensors="pt"), max_new_tokens=32)
    results_reordered[context] = time.time() - t0
    print(context, results_reordered[context])

print()
slowest_config = max(results_reordered, key=results_reordered.get)
print(f"Slowest config this run: {slowest_config} (ran 1st)")
print("The 'slowest' result moved to whichever config ran first —")
print("that confirms it's a loop-position effect, not a real config property.")


2048 1.2803125381469727
512 0.10070085525512695
128 0.05906224250793457

Slowest config this run: 2048 (ran 1st)
The 'slowest' result moved to whichever config ran first —
that confirms it's a loop-position effect, not a real config property.


In [5]:
if MODE == "simulate":
    FakeModel.reset_session()  # fresh session

# throwaway warm-up, discarded, absorbs the one-time cost before real timing
_ = model.generate(**tok(prompt_of_len(64), return_tensors="pt"), max_new_tokens=8)

results = {}
for context in [128, 512, 2048]:
    prompt = prompt_of_len(context)
    t0 = time.time()
    out = model.generate(**tok(prompt, return_tensors="pt"), max_new_tokens=32)
    results[context] = time.time() - t0
    print(context, results[context])


128 0.05953359603881836
512 0.0992124080657959
2048 0.26479125022888184


In [6]:
assert results[128] < results[512] < results[2048], (
    f"expected latency to climb with context length, got {results}"
)
print("GREEN CHECK: PASS")


GREEN CHECK: PASS
